visualization of trading 
CSV файл генерируется в sessions_conditions.ipynb 

In [1]:
# Динамический импорт и инициализация библиотек и путей для работы с данными в Python <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
from pathlib import Path
import csv
import sys
import os
import pandas as pd
import json

sub_project_dir = "trading_conditions"

file_dir = os.getcwd()                                                              # Определяем путь к текущему файлу (где выполняется код)
print(f"Файл в директории:                                      {file_dir}")

project_dir                     = Path(file_dir).parent.parent                      # Переход на уровень выше (fc_to_mt5_migrations/own_platform)
print(f"Рабочая директория проекта:                             {project_dir}")

parent_dir                      = Path.cwd().parent.parent.parent                   # Переход на уровень выше (fc_to_mt5_migrations)
print(f"Рабочая директория проекта для доступа к библиотекам:   {parent_dir}")

directory_data_log_files        = os.path.join(project_dir, sub_project_dir, 'log_data_files')
print(f"[directory_data_log_files];     Путь к каталогу с лог-файлами:                          {directory_data_log_files}")

directory_data_temp_files       = os.path.join(project_dir, sub_project_dir, 'working_data_files') 
print(f"[directory_data_temp_files];    Путь к каталогу с временными файлами:                   {directory_data_temp_files}")

directory_data_original_data    = os.path.join(project_dir, 'original_data')        # Путь к каталогу с оригинальными данными
print(f"Путь к каталогу с оригинальными данными:                {directory_data_original_data}")

directory_data_set              = os.path.join(project_dir, 'data_set')             # Путь к каталогу с конфигурационными данными
print(f"Путь к каталогу с файлами настроек:                     {directory_data_set}")

directory_data_output           = os.path.join(parent_dir, 'output_data')           # Путь к каталогу с выходными данными

libraries_path = os.path.join(parent_dir, "libraries_py")                           # Формируем путь к libraries_py каталогу с библиотеками *.py
sys.path.append(libraries_path)                                                     # sys.path — это список путей, где Python ищет модули при import module_name.

if libraries_path in sys.path: print(f"✅ Каталог {libraries_path} успешно добавлен в sys.path")
else: print(f"❌ Ошибка: {libraries_path} не найден в sys.path")

# Динамически импорт необходимых функций <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
file_imports = "dynamic_import_functions.py"                                        # Библиотека для динамического импорта
file_imports_path = os.path.join(libraries_path, file_imports)
if os.path.exists(file_imports_path):
    import importlib
    importlib.invalidate_caches()                                                   # Сбрасываем кэш перед импортом
    from dynamic_import_functions import import_functions, print_import_function_info
    print(f"\n ✅ Импорт [{file_imports}] успешен.")
else:
    print(f"\n ERROR: Файл '{file_imports}' не найден по пути {file_imports_path}, импорт не выполнен.\n")

modules_to_import = {                                                               # Формируем словарь, с именами файлов и функциями в них
    "yar_sed_general_lib":
        [libraries_path,
                "pd_set_option",                     # Вывод ДФ
                "df_to_csv",                         # Сохранение ДФ в CSV 
                "CSVLoader",
                #$"save_data_log_work_file",
                #"detect_encoding",
                #"time_to_minutes",
                #"load_string_list",
                #"list_print",
                #"move_column",
                #"filter_df_by_suffix",
                #"check_columns_exist_id_df",
                #"merge_left_with_check"        # Проверка наличия колонок в DataFrame 
                ],           
    #"sql_request_2": [libraries_path, "pd_read_sql","get_sql_tab"],
                    }

imported = import_functions(modules_to_import)          # Импортируем модули из словаря modules_to_import

print_import_function_info(modules_to_import, imported) # Выводим переменные ожидаемые импортированными функциями 

Файл в директории:                                      c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\ipynb_files
Рабочая директория проекта:                             c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform
Рабочая директория проекта для доступа к библиотекам:   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations
[directory_data_log_files];     Путь к каталогу с лог-файлами:                          c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\log_data_files
[directory_data_temp_files];    Путь к каталогу с временными файлами:                   c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files
Путь к каталогу с оригинальными данными:                c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\original_data
Путь к каталогу с файлами настроек:                 

In [ ]:
# ЯЧЕЙКА С ФУНКЦИЯМИ ИСПОЛЬЗУЕМЫМИ НЕПОСРЕДСТВЕННО В ЭТОМ ФАЙЛЕ <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
"""'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''"""

# [ФУНКЦИЯ] Сохранение DataFrame в HTML с красивым форматированием, переносами строк и без обрезания <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
# Предполагаем, что ваш DataFrame уже есть, например problems_df
# Если хотите сохранить clean_df — просто замените имя переменной

def df_to_html(df_to_save, table_title, name_file):
    # 1. Форматируем колонку example_patterns (если она есть)
    if 'example_patterns' in df_to_save.columns:
        def pretty_patterns(patterns):
            if not patterns:
                return "Нет шаблонов"
            return "<br><br>".join(
                json.dumps(p, indent=2, ensure_ascii=False).replace('\n', '<br>')
                for p in patterns
            )
        
        df_to_save = df_to_save.copy()
        df_to_save['example_patterns'] = df_to_save['example_patterns'].apply(pretty_patterns)

    # 2. Создаём стиль
    styled = df_to_save.style \
        .set_properties(**{
            'text-align': 'left',
            'white-space': 'pre-wrap',
            'font-family': 'Consolas, monospace',
            'font-size': '13px',
            'padding': '6px',
            'border': '1px solid #ddd'
        }) \
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#e6f0ff'),
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]},
            {'selector': 'caption', 'props': [
                ('caption-side', 'top'),
                ('font-size', '16px'),
                ('font-weight', 'bold'),
                ('padding', '10px')
            ]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]}
        ]) \
        .set_caption(table_title) \
        .hide(axis='index')  # убираем индексы, если не нужны

    # 3. Сохраняем в HTML
    output_file = f"{name_file}.html"
    styled.to_html(
        output_file,
        index=False,          # без индекса строк
        escape=False,         # важно — чтобы <br> и теги работали
        classes='table table-striped table-bordered'
    )

    # 4. Выводим полный путь к файлу
    full_path = os.path.abspath(output_file)
    print(f"HTML-файл успешно сохранён:")
    print(full_path)

    print("\nОткройте этот файл в любом браузере — таблица будет красивая, с переносами строк и без обрезания.")

# [ФУНКЦИЯ] Проверка согласованности значений по дням недели внутри групп name_market <<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
# ''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''
def check_day_consistency(df, market_col='name_market'):
    """
    Проверяет согласованность значений по дням недели внутри групп name_market.
    
    Возвращает три DataFrame:
    - stats_df: статистика по всем группам
    - problems_df: только группы с коллизиями (unique_patterns_count > 1)
    - clean_df: группы без коллизий (unique_patterns_count == 1)
    """
    day_cols = [
        'open_Sun', 'close_Sun',
        'open_Mon', 'close_Mon',
        'open_Tue', 'close_Tue',
        'open_Wed', 'close_Wed',
        'open_Thu', 'close_Thu',
        'open_Fri', 'close_Fri',
        'open_Sat', 'close_Sat'
    ]
    
    relevant_cols = [market_col] + day_cols
    
    # Группируем
    grouped = df[relevant_cols].groupby(market_col)
    
    stats = []
    problems = []
    clean_groups = []
    
    for market, group in grouped:
        unique_patterns = group[day_cols].drop_duplicates()
        num_patterns = len(unique_patterns)
        
        row_count = len(group)
        
        # Статистика
        stats.append({
            market_col: market,
            'rows_in_group': row_count,
            'unique_day_patterns': num_patterns
        })
        
        if num_patterns > 1:
            # Проблемная группа
            problems.append({
                market_col: market,
                'rows_in_group': row_count,
                'unique_patterns_count': num_patterns,
                'example_patterns': unique_patterns.head(3).to_dict('records')
            })
        else:
            # Чистая группа — сохраняем как есть (или можно взять только одну строку)
            clean_groups.append(group)
    
    # Финальные DataFrame
    stats_df = pd.DataFrame(stats)
    
    problems_df = pd.DataFrame(problems) if problems else pd.DataFrame(
        columns=[market_col, 'rows_in_group', 'unique_patterns_count', 'example_patterns']
    )
    
    # Если есть чистые группы — объединяем их в один DataFrame
    clean_df = pd.concat(clean_groups) if clean_groups else pd.DataFrame()
    
    return stats_df, problems_df, clean_df

In [2]:


def prepare_sessions_for_viz(df_sessions_all):
    """
    Преобразует вертикальный формат сессий (несколько строк на символ/день)
    в широкий формат (одна строка на символ), объединяя множественные сессии через <br>.
    """
    df = df_sessions_all.copy()
    
    # Создаем строковое представление интервала "HH:MM-HH:MM"
    df['interval'] = df['open'] + '-' + df['close']
    
    # Группируем по символу и дню, склеивая интервалы, если их несколько в сутках
    # Результат: "00:00-19:59 <br> 21:00-23:59"
    day_grouped = df.groupby(['symbolId', 'cluster', 'day'])['interval'].apply(lambda x: '<br>'.join(x)).reset_index()
    
    # Разворачиваем дни в колонки (Pivot)
    df_wide = day_grouped.pivot(index=['symbolId', 'cluster'], columns='day', values='interval').reset_index()
    
    # Переименовываем колонки в понятные дни недели
    days_map = {0: 'Sun', 1: 'Mon', 2: 'Tue', 3: 'Wed', 4: 'Thu', 5: 'Fri', 6: 'Sat'}
    df_wide.columns = [days_map.get(col, col) for col in df_wide.columns]
    
    # Добавляем префиксы open/close для совместимости со старой функцией (опционально)
    # Или просто оставляем имена дней. Давайте сделаем имена дней для чистоты.
    return df_wide.fillna('Closed')

# ПРИМЕНЕНИЕ:
# Предположим, df_sessions_all получен из вашей функции create_sessions_dfs
#df_viz = prepare_sessions_for_viz(df_sessions_all)


def check_day_consistency_new(df, group_col='cluster'):
    """
    Версия для нового формата: проверяет, что у всех символов внутри кластера 
    одинаковые текстовые паттерны сессий по дням недели.
    """
    day_cols = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
    # Проверяем только те колонки, которые реально есть в DF
    existing_days = [c for c in day_cols if c in df.columns]
    
    grouped = df.groupby(group_col)
    
    stats, problems, clean_groups = [], [], []
    
    for name, group in grouped:
        # Ищем уникальные комбинации расписаний
        unique_patterns = group[existing_days].drop_duplicates()
        num_patterns = len(unique_patterns)
        row_count = len(group)
        
        stats.append({
            group_col: name,
            'symbols_count': row_count,
            'unique_patterns': num_patterns
        })
        
        if num_patterns > 1:
            problems.append({
                group_col: name,
                'symbols_count': row_count,
                'unique_patterns_count': num_patterns,
                'example_patterns': unique_patterns.head(3).to_dict('records')
            })
        else:
            clean_groups.append(group)
            
    stats_df = pd.DataFrame(stats)
    problems_df = pd.DataFrame(problems) if problems else pd.DataFrame()
    clean_df = pd.concat(clean_groups) if clean_groups else pd.DataFrame()
    
    return stats_df, problems_df, clean_df



def df_to_html(df_to_save, table_title, name_file):
    # 1. Форматируем колонку example_patterns (если она есть)
    # Это нужно для красивого отображения различий в "проблемных" группах
    if 'example_patterns' in df_to_save.columns:
        def pretty_patterns(patterns):
            if not patterns:
                return "Нет шаблонов"
            return "<br><br>".join(
                json.dumps(p, indent=2, ensure_ascii=False).replace('\n', '<br>')
                for p in patterns
            )
        
        df_to_save = df_to_save.copy()
        df_to_save['example_patterns'] = df_to_save['example_patterns'].apply(pretty_patterns)

    # 2. Создаём стиль (добавлено vertical-align для красоты многострочных сессий)
    styled = df_to_save.style \
        .set_properties(**{
            'text-align': 'left',
            'vertical-align': 'top',      # ← Добавлено: чтобы текст начинался сверху ячейки
            'white-space': 'pre-wrap',    # Поддерживает переносы строк <br>
            'font-family': 'Consolas, monospace',
            'font-size': '13px',
            'padding': '6px',
            'border': '1px solid #ddd'
        }) \
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#e6f0ff'),
                ('font-weight', 'bold'),
                ('text-align', 'center'),
                ('vertical-align', 'middle')
            ]},
            {'selector': 'caption', 'props': [
                ('caption-side', 'top'),
                ('font-size', '16px'),
                ('font-weight', 'bold'),
                ('padding', '10px'),
                ('color', '#333')
            ]},
            {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]}
        ]) \
        .set_caption(table_title) \
        .hide(axis='index') 

    # 3. Сохраняем в HTML
    output_file = f"{name_file}.html"
    
    # Используем render() для получения HTML-строки из Styler объекта
    html_output = styled.to_html(
        escape=False, # КРИТИЧНО: позволяет отображать <br> как перенос строки, а не как текст
        index=False,
        classes='table table-striped table-bordered'
    )

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_output)

    # 4. Выводим полный путь к файлу
    full_path = os.path.abspath(output_file)
    print(f"HTML-файл успешно сохранён:")
    print(full_path)
    print("\nТаблица адаптирована под многострочные сессии. Используйте браузер для просмотра.")

In [5]:
file_path_symbols = Path(directory_data_temp_files)  / "df_sessions_enriched.csv"                       # Файл с торговыми сесcиями
print("latest_symbolTicks_df.csv:", file_path_symbols)
loader = imported["CSVLoader"](file_path_symbols, delimiter=',', encoding='utf-8', df_name='my_dataframe')  # CSVLoader для загрузки данных из файла
sessions_for_visualization_df = loader.load_data()

df = sessions_for_visualization_df.copy()
df = df.loc[:, ~df.columns.str.startswith("1_")]                    # Удаляем колонки, начинающиеся с "1_"; в последствии вернём когда настроим окна
df.columns = [col.removeprefix("0_") for col in df.columns]         # Удаляем префикс "0_" из имен колонок
df.columns = df.columns.str.replace(r'_([0-6])$', lambda m: '_' + ['Sun','Mon','Tue','Wed','Thu','Fri','Sat'][int(m.group(1))], regex=True)
day_list = ['open_Sun', 'close_Sun', 'open_Mon', 'close_Mon', 'open_Tue', 'close_Tue', 'open_Wed', 'close_Wed', 'open_Thu', 'close_Thu', 'open_Fri', 'close_Fri', 'open_Sat', 'close_Sat']
df = df[df['tradeMode_s'] == 4]

imported["pd_set_option"]("df", df, 5)


# Предполагаем, что df — ваш DataFrame
stats_df, problems_df, clean_df = check_day_consistency_viz(df, market_col='name_market') # [ФУНКЦИЯ] Проверка согласованности значений по дням недели внутри групп name_market

#print("=== Статистика по всем группам ===")
#display(stats_df.sort_values('unique_day_patterns', ascending=False))

print("\n=== Проблемные группы (коллизии) ===")
with pd.option_context('display.max_colwidth', None):
    display(problems_df)
unique_values = problems_df['name_market'].unique().tolist() # Получить список уникальных значений (в порядке появления)

df_filtered = df[df['name_market'].isin(unique_values)].copy()   # 1. Фильтруем строки, где значение колонки входит в список

#grouped = df_filtered.groupby('name_market')    # 2. Группируем по этой колонке (можно сразу посчитать что-то полезное)
#df_grouped = grouped.size().reset_index(name='count')
#display(grouped.head(20))                      # 3. Выводим первые 20 строк каждой группы для проверки
df_to_save = df_filtered.copy()
imported["df_to_csv"](df_to_save,      "Группы c разным торговым временем по символам.csv")  # Сохранение df_to_save в CSV файл
df_to_html(df_to_save, "Группы c разным торговым временем по символам", "problems_df")


print("\n=== Группы без коллизий (все строки внутри name_market имеют одинаковые дни недели) ===")
print(f"Количество чистых групп: {clean_df['name_market'].nunique()}")
print(f"Количество строк в чистых группах: {len(clean_df)}")
print("\n=== ЧИСТЫЕ группы ===")
with pd.option_context('display.max_colwidth', None):
    # Удаляем полные дубликаты строк
    clean_df_unique = clean_df.drop_duplicates()
    print(f"Исходное количество строк: {len(clean_df)}")
    print(f"После удаления дубликатов: {len(clean_df_unique)}")
    print(f"Количество уникальных name_market: {clean_df_unique['name_market'].nunique()}")
    display(clean_df_unique)
    
df_to_save = clean_df_unique  # ← или clean_df, или stats_df
imported["df_to_csv"](df_to_save,      "Группы без коллизий (все строки внутри name_market имеют одинаковые дни недели).csv")  # Сохранение df_to_save в CSV файл
df_to_html(df_to_save, "Группы без коллизий (все строки внутри name_market имеют одинаковые дни недели)", "clean_df_unique")


# Если нужно посмотреть сами данные чистых групп
# display(clean_df.head(20))

latest_symbolTicks_df.csv: c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files\df_sessions_enriched.csv
[class CSVLoader]: DataFrame 'my_dataframe' успешно создан из 'c:\unique_data\rep_fo_metatrader_server\fc_to_mt5_migrations\own_platform\trading_conditions\working_data_files\df_sessions_enriched.csv'.

df  (1,614 строк × 14 колонок)


,name_s,displayName_s,marketId_s,name_market,tradeMode_s,symbolId,0,1,2,3,4,5,6,id
0,AUDCAD,AUD / CAD,2.0,Minor,4.0,1,21:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-23:59,00:00-20:59,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1950,AU.N,Anglo Gold,33.0,CFDs - Stocks United States,4.0,10213,NaN,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,13:30-19:59,NaN,33.0


NameError: name 'check_day_consistency_viz' is not defined

In [ ]:
from collections import defaultdict

def check_day_consistency(df, market_col='name_market'):
    """
    Проверяет, что для каждого уникального name_market
    все строки имеют одинаковые значения по колонкам дней недели.
    
    Возвращает:
    - DataFrame с проблемными группами (если есть)
    - Статистику: сколько уникальных наборов дней на каждое name_market
    """
    day_cols = ['open_Sun', 'close_Sun','open_Mon', 'close_Mon','open_Tue', 'close_Tue','open_Wed', 'close_Wed','open_Thu', 'close_Thu','open_Fri', 'close_Fri','open_Sat', 'close_Sat']
    relevant_cols = [market_col] + day_cols             # Только нужные колонки
    grouped = df[relevant_cols].groupby(market_col)     # Группируем по name_market и считаем уникальные комбинации дней
    
    stats = []
    problems = []
    
    for market, group in grouped:
        # Уникальные наборы значений по дням
        unique_patterns = group[day_cols].drop_duplicates()
        num_patterns = len(unique_patterns)
        
        stats.append({
            market_col: market,
            'unique_day_patterns': num_patterns,
            'rows_in_group': len(group)
        })
        
        if num_patterns > 1:
            # Есть несогласованность
            problems.append({
                market_col: market,
                'rows_in_group': len(group),
                'unique_patterns_count': num_patterns,
                'example_patterns': unique_patterns.head(3).to_dict('records')  # первые 3 разных
            })
    
    stats_df = pd.DataFrame(stats)
    problems_df = pd.DataFrame(problems) if problems else pd.DataFrame()
    
    return stats_df, problems_df


# Предполагаем, что df — ваш DataFrame
stats_df, problems_df = check_day_consistency(df, market_col='name_market')


import json

def format_patterns(patterns):
    if not patterns:
        return "Нет шаблонов"
    
    formatted = []
    for i, p in enumerate(patterns, 1):
        # Превращаем словарь в одну строку без лишних переносов
        one_line = json.dumps(p, separators=(', ', ': '), ensure_ascii=False)
        formatted.append(f"Шаблон {i}: {one_line}")
    
    return "\n".join(formatted)


# Применяем ко всей таблице
temp_df = problems_df.copy()
temp_df['example_patterns'] = temp_df['example_patterns'].apply(format_patterns)

# Выводим красиво
with pd.option_context(
    'display.max_colwidth', None,
    'display.max_rows', None,
    'display.width', 1200
):
    display(temp_df.style
        .set_properties(**{
            'text-align': 'left',
            'white-space': 'pre-wrap',
            'font-family': 'Consolas, monospace',
            'font-size': '13px',
            'padding': '6px',
            'border': '1px solid #ccc'
        })
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#e6f0ff'), ('font-weight', 'bold')]},
            {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-size', '16px'), ('font-weight', 'bold')]}
        ])
        .set_caption("Проблемные группы name_market (значения дней недели НЕ совпадают)")
    )


pd.set_option('display.max_colwidth', None)
imported["pd_set_option"]("problems_df", problems_df, 50)
